# Held-out HSV-2 population validation

This notebook audits exact guide/PAM retention in public HSV-2 records excluded from discovery. It separates an observable locus difference from a locus missing in a partial record. These are population-genomic results, not editing, safety, efficacy, delivery, or therapeutic evidence.

## Reproduce

Install the optional mapper with `python -m pip install -e '.[population]'`, then run `bash scripts/run_hsv2_population_validation.sh`. Existing NCBI downloads and completed outputs are reused.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
POPULATION = ROOT / 'reports' / 'hsv2_population_heldout'
REPORT = ROOT / 'reports' / 'hsv2_population_report_balanced'
required = [POPULATION / 'population_manifest.json', REPORT / 'population_report_manifest.json', REPORT / 'candidate_population_comparison.csv', REPORT / 'gene_population_summary.csv']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Run the population workflow first. Missing: ' + ', '.join(missing))

## Panel audit

The QC table makes exclusions visible. Valid IUPAC ambiguity is counted against the declared threshold; exact duplicates and every discovery accession are excluded.

In [ ]:
panel_manifest = json.loads((POPULATION / 'population_manifest.json').read_text(encoding='utf-8'))
qc = pd.read_csv(POPULATION / 'population_qc.csv')
display(panel_manifest)
display(qc.groupby(['decision', 'reason'], dropna=False).size().rename('record_count').reset_index())

## Locus-aware denominator

A guide is called different only when a high-quality whole-record alignment covers its reference interval and the exact protospacer plus compatible PAM is absent. Otherwise the record is unresolved for that locus.

In [ ]:
locus_manifest = json.loads((REPORT / 'population_report_manifest.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(REPORT / 'candidate_population_comparison.csv')
display(locus_manifest)
display(comparison['population_validation_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='candidate_count'))

## Focus genes

Population support is displayed as an independent evidence axis and is not added to targetability, essentiality, or predicted-disruption scores.

In [ ]:
focus_genes = ['UL3', 'UL10', 'UL18', 'UL20', 'UL36', 'UL52', 'UL53', 'UL19', 'UL30']
genes = pd.read_csv(REPORT / 'gene_population_summary.csv')
focus = genes[genes['gene_name'].isin(focus_genes)].copy()
focus['fully_supported_fraction'] = focus['exact_in_all_observable_records_candidate_count'] / focus['locus_evaluable_unique_candidate_count']
display(focus[['gene_name', 'locus_evaluable_unique_candidate_count', 'exact_in_all_observable_records_candidate_count', 'median_observable_locus_exact_target_coverage', 'minimum_observable_locus_exact_target_coverage', 'best_population_supported_candidate_id']])
focus.sort_values('fully_supported_fraction').plot.barh(x='gene_name', y='fully_supported_fraction', legend=False, figsize=(9, 5), title='Exact target support in observable held-out loci')
plt.xlim(0, 1)
plt.xlabel('Fraction of evaluable guides exact in every observable record')
plt.show()

## Candidate-level audit

Reference-multi-copy guides are explicitly flagged because an exact occurrence elsewhere cannot be attributed safely to one locus.

In [ ]:
columns = ['post_human_rank', 'candidate_id', 'mapped_gene_names', 'reference_unique_target', 'locus_observable_record_count', 'exact_target_in_observable_locus_count', 'observable_locus_exact_target_coverage', 'locus_unresolved_record_count', 'population_validation_status']
display(comparison.sort_values('post_human_rank')[columns].head(30))